<a href="https://colab.research.google.com/github/MaineCalabrezi13/DiagnosticoPlantas/blob/main/doen%C3%A7as_folhas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import shutil
import random

In [ ]:
# 1. Configurar a credencial do Kaggle direto no ambiente do Colab
# Colar o token dentro das aspas abaixo
os.environ['KAGGLE_API_TOKEN'] = "KGAT_78d11ff3ddf72ec99f8d02cb4d33f2cc"

# 2. Baixar o dataset direto pelo link do Kaggle
!kaggle datasets download -d emmarex/plantdisease

# 3. Descompactar o arquivo direto no ambiente do Colab
!unzip -q plantdisease.zip -d meu_dataset
print("Prontinho! O dataset foi baixado e descompactado com sucesso.")

Dataset URL: https://www.kaggle.com/datasets/emmarex/plantdisease
License(s): unknown
100% 658M/658M [00:04<00:00, 168MB/s]

Prontinho! O dataset foi baixado e descompactado com sucesso.


In [ ]:
print("--- Conteúdo da pasta PlantVillage (Maiúscula) ---") # Esse é o que vamos usar
try:
    print(os.listdir("meu_dataset/PlantVillage")[:5]) # Mostra os primeiros 5 itens
except Exception as e:
    print("Erro:", e)

print("\n--- Conteúdo da pasta plantvillage (Minúscula) ---")
try:
    print(os.listdir("meu_dataset/plantvillage")[:5]) # Mostra os primeiros 5 itens
except Exception as e:
    print("Erro:", e)

--- Conteúdo da pasta PlantVillage (Maiúscula) ---
['Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy', 'Potato___Late_blight', 'Tomato_Leaf_Mold']

--- Conteúdo da pasta plantvillage (Minúscula) ---
['PlantVillage']


In [ ]:
# 1. Caminhos de origem e destino
origem_dir = "meu_dataset/PlantVillage"
destino_base = "dataset_yolo"

# Criar a estrutura de pastas do YOLO
os.makedirs(os.path.join(destino_base, "train"), exist_ok=True)
os.makedirs(os.path.join(destino_base, "val"), exist_ok=True)

# Pegar todas as categorias de doenças/plantas
categorias = os.listdir(origem_dir)

# Proporção: 80% das fotos para treinar o modelo e 20% para testar (validação)
split_proporcao = 0.8

print("Organizando as imagens... Aguarde um momento.")

for cat in categorias:
    caminho_cat = os.path.join(origem_dir, cat)

    # Ignorar arquivos que não sejam pastas
    if not os.path.isdir(caminho_cat):
        continue

    # Criar as pastas da categoria específica dentro de train e val
    os.makedirs(os.path.join(destino_base, "train", cat), exist_ok=True)
    os.makedirs(os.path.join(destino_base, "val", cat), exist_ok=True)

    # Listar e embaralhar todas as imagens daquela doença
    imagens = os.listdir(caminho_cat)
    random.shuffle(imagens)

    # Calcular o ponto de divisão
    limite = int(len(imagens) * split_proporcao)

    imagens_train = imagens[:limite]
    imagens_val = imagens[limite:]

    # Copiar os arquivos para os seus devidos lugares
    for img in imagens_train:
        shutil.copy(os.path.join(caminho_cat, img), os.path.join(destino_base, "train", cat, img))

    for img in imagens_val:
        shutil.copy(os.path.join(caminho_cat, img), os.path.join(destino_base, "val", cat, img))

print("Sucesso! Seus dados foram divididos e organizados na pasta 'dataset_yolo'.")

Organizando as imagens... Aguarde um momento.
Sucesso! Seus dados foram divididos e organizados na pasta 'dataset_yolo'.


In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.2 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

# 1. Carrega o modelo YOLO específico para classificação (versão leve 'nano')
model = YOLO('yolov8n-cls.pt')

# 2. Inicia o treinamento com os seus dados ordenados
# Passamos o caminho da pasta base onde estão 'train' e 'val'
results = model.train(
    data='dataset_yolo',
    epochs=100,
    imgsz=320,
    degrees=10,
    scale=0.2,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4
)
print("TREINAMENTO CONCLUÍDO COM SUCESSO!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_yolo, degrees=10, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=Fa